# SQL Server -> Pre-Ingestion Analysis

Profiles a SQL Server database **before** you ingest it, and emits the
`(database, schema, table)` config tuples consumed by `sqlserver_to_snowflake.py`.

**What it does**
1. Connects to SQL Server (Windows/trusted auth, or a SQL login).
2. Lists every `BASE TABLE` in the database you point it at.
3. Pulls the **column inventory** + a **Top N** row sample for each table.
4. Flags likely **incremental / change-tracking** columns (update dates, `rowversion`, ...).
5. Writes an Excel workbook: a **Summary** tab + **one tab per table** (column counts, types, sample rows).
6. Prints the ready-to-paste `tables = [ ("DB","schema","table"), ... ]` config.

Run for **all** tables or just a chosen few. Edit the single **Config** cell below, then *Run All*.

## 0 - Setup
`pip install pyodbc openpyxl` once if needed.

In [ ]:
from pathlib import Path
from datetime import datetime

# ---------------- connection ----------------
SERVER   = "teloutdbp002.asia.tel.com"
DATABASE = "TEL_GWS"                  # the DB to profile (drives INFORMATION_SCHEMA + the queries)
USERNAME = "DevOpsUser-TEH-Sel-DB2"   # leave "" / None for Windows (trusted) auth
PASSWORD = ""
DRIVER   = "{SQL Server Native Client 11.0}"

# ---------------- analysis options ----------------
TOP_N              = 100        # sample rows pulled per table
INCLUDE_ROW_COUNTS = True       # exact COUNT(*) per table (set False if tables are huge/slow)

# Which tables?
#   "ALL"   -> every base table in the DB
#   or a list:  ["dbo.gws_category", "gws_other_table", "sales.orders"]
#               (schema is optional; "table" alone matches that table in any schema)
TABLES = "ALL"

# ---------------- output ----------------
OUTPUT_DIR  = Path(".")
OUTPUT_FILE = OUTPUT_DIR / f"pre_ingestion_analysis_{DATABASE}_{datetime.now():%Y%m%d_%H%M}.xlsx"

# ---------------- incremental-column heuristics (tweak freely) ----------------
INCREMENTAL_NAME_PATTERNS = [
    r"updat", r"modif", r"chang", r"creat", r"insert", r"load", r"last",
    r"timestamp", r"rowversion", r"row_version", r"version", r"audit",
    r"edit", r"sync", r"etl", r"_ts$", r"_dt$", r"date",
]
STRONG_INCREMENTAL_TYPES = {"rowversion", "timestamp"}   # SQL Server rowversion = perfect delta key
DATETIME_TYPES = {"datetime", "datetime2", "smalldatetime", "datetimeoffset", "date", "time"}

print("DB          :", DATABASE)
print("Tables      :", TABLES)
print("Top N       :", TOP_N)
print("Row counts  :", INCLUDE_ROW_COUNTS)
print("Output file :", OUTPUT_FILE)

## 1 - Connection & query helpers
Same pyodbc connection style as `sqlserver_to_snowflake.py` (including the `datetimeoffset` output converter).

In [ ]:
import re, struct, datetime as dt
from datetime import timedelta, timezone
import pyodbc

def _handle_datetimeoffset(value):
    y, mo, d, h, mi, s, frac_ns, tz_min = struct.unpack("<hhhhhhIh", value[:18])
    return dt.datetime(y, mo, d, h, mi, s, frac_ns // 1000,
                       tzinfo=timezone(timedelta(minutes=tz_min))).isoformat()

def connect():
    trusted = "no" if USERNAME else "yes"
    kwargs = {"Server": SERVER, "Database": DATABASE, "Trusted_Connection": trusted}
    if USERNAME:
        kwargs["user"] = USERNAME
        kwargs["password"] = PASSWORD
    conn = pyodbc.connect(f"Driver={DRIVER};", **kwargs)
    conn.add_output_converter(-155, _handle_datetimeoffset)
    return conn

def list_tables(conn):
    cur = conn.cursor()
    cur.execute(f"""
        SELECT TABLE_SCHEMA AS schema_name, TABLE_NAME AS table_name
        FROM [{DATABASE}].INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
        ORDER BY TABLE_SCHEMA, TABLE_NAME
    """)
    out = [(r.schema_name, r.table_name) for r in cur.fetchall()]
    cur.close()
    return out

def get_columns(conn, schema, table):
    cur = conn.cursor()
    cur.execute(f"""
        SELECT COLUMN_NAME, DATA_TYPE, ORDINAL_POSITION, IS_NULLABLE,
               CHARACTER_MAXIMUM_LENGTH, NUMERIC_PRECISION, NUMERIC_SCALE
        FROM [{DATABASE}].INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = ? AND TABLE_NAME = ?
        ORDER BY ORDINAL_POSITION
    """, schema, table)
    cols = []
    for r in cur.fetchall():
        cols.append({
            "ordinal": r.ORDINAL_POSITION, "name": r.COLUMN_NAME, "type": r.DATA_TYPE,
            "nullable": r.IS_NULLABLE, "max_len": r.CHARACTER_MAXIMUM_LENGTH,
            "precision": r.NUMERIC_PRECISION, "scale": r.NUMERIC_SCALE,
        })
    cur.close()
    return cols

def get_row_count(conn, schema, table):
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM [{DATABASE}].[{schema}].[{table}]")
    n = int(cur.fetchone()[0]); cur.close()
    return n

def get_top_n(conn, schema, table, n):
    cur = conn.cursor()
    cur.execute(f"SELECT TOP ({int(n)}) * FROM [{DATABASE}].[{schema}].[{table}]")
    headers = [d[0] for d in cur.description]
    rows = [list(r) for r in cur.fetchall()]
    cur.close()
    return headers, rows

def length_or_precision(col):
    if col["max_len"] is not None:
        return "max" if col["max_len"] == -1 else str(col["max_len"])
    if col["precision"] is not None:
        return f'{col["precision"]},{col["scale"] or 0}'
    return ""

def classify_incremental(col):
    name = (col["name"] or "").lower()
    dtype = (col["type"] or "").lower()
    name_hit = any(re.search(p, name) for p in INCREMENTAL_NAME_PATTERNS)
    if dtype in STRONG_INCREMENTAL_TYPES:
        return "High", f"{dtype} - ideal change-tracking key"
    if name_hit and dtype in DATETIME_TYPES:
        return "High", f"name match + {dtype}"
    if dtype in DATETIME_TYPES:
        return "Medium", f"datetime column ({dtype})"
    if name_hit:
        return "Low", "name pattern only"
    return None, None

def select_tables(all_tables):
    if isinstance(TABLES, str) and TABLES.strip().upper() == "ALL":
        return list(all_tables)
    wanted = []
    for entry in TABLES:
        e = str(entry).strip()
        if "." in e:
            sch, tbl = e.split(".", 1); wanted.append((sch.lower(), tbl.lower()))
        else:
            wanted.append((None, e.lower()))
    def matches(sch, tbl, ws, wt):
        return tbl.lower() == wt and (ws is None or ws == sch.lower())
    picked = [(sch, tbl) for sch, tbl in all_tables
              if any(matches(sch, tbl, ws, wt) for ws, wt in wanted)]
    missing = [f"{(ws + '.') if ws else ''}{wt}" for ws, wt in wanted
               if not any(matches(sch, tbl, ws, wt) for sch, tbl in all_tables)]
    if missing:
        print("WARNING - requested but not found in DB:", ", ".join(missing))
    return picked

print("helpers ready")

## 2 - Discover tables

In [ ]:
conn = connect()
all_tables = list_tables(conn)
selected = select_tables(all_tables)

print(f"{len(all_tables)} base tables in [{DATABASE}]; analysing {len(selected)}:\n")
for sch, tbl in selected:
    print(f"  {sch}.{tbl}")

## 3 - Profile each table
Pulls columns + a Top N sample and flags incremental candidates. A locked/unreadable table is
skipped with a note recorded in the workbook, so one failure won't stop the whole run.

In [ ]:
analysis = []
for sch, tbl in selected:
    rec = {"schema": sch, "table": tbl, "error": None, "sample_error": None,
           "columns": [], "row_count": None, "headers": [], "sample": []}
    try:
        cols = get_columns(conn, sch, tbl)
        for c in cols:
            c["inc_conf"], c["inc_why"] = classify_incremental(c)
        rec["columns"] = cols
        if INCLUDE_ROW_COUNTS:
            rec["row_count"] = get_row_count(conn, sch, tbl)
        try:
            rec["headers"], rec["sample"] = get_top_n(conn, sch, tbl, TOP_N)
        except Exception as e:
            rec["sample_error"] = str(e)
    except Exception as e:
        rec["error"] = str(e)
    analysis.append(rec)

    n_inc = sum(1 for c in rec["columns"] if c.get("inc_conf"))
    if rec["error"]:
        status = "ERROR: " + rec["error"]
    else:
        rows = rec["row_count"] if rec["row_count"] is not None else "?"
        status = f'{len(rec["columns"])} cols, {n_inc} incremental, {rows} rows'
    print(f'  {sch}.{tbl:<42} {status}')

print("\nprofiling done.")

## 4 - Write the Excel workbook
A **Summary** tab plus **one tab per table**: the column inventory (incremental candidates
highlighted green) followed by the Top N sample rows.

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FONT = "Calibri"
NAVY      = "1F3864"
HEAD_FILL = PatternFill("solid", fgColor="2E5496")
BANNER    = PatternFill("solid", fgColor="D9E1F2")
ZEBRA     = PatternFill("solid", fgColor="F2F5FB")
INC_HIGH  = PatternFill("solid", fgColor="C6EFCE")
INC_MED   = PatternFill("solid", fgColor="FFEB9C")
INC_LOW   = PatternFill("solid", fgColor="F2F2F2")
ERR_FILL  = PatternFill("solid", fgColor="FFC7CE")
CONF_FILL = {"High": INC_HIGH, "Medium": INC_MED, "Low": INC_LOW}

TITLE_FONT = Font(name=FONT, bold=True, size=15, color=NAVY)
NOTE_FONT  = Font(name=FONT, italic=True, size=9, color="808080")
HEAD_FONT  = Font(name=FONT, bold=True, size=11, color="FFFFFF")
LABEL_FONT = Font(name=FONT, bold=True, size=11, color=NAVY)
BASE_FONT  = Font(name=FONT, size=10)
BOLD_SMALL = Font(name=FONT, bold=True, size=10)
LINK_FONT  = Font(name=FONT, size=10, color="0563C1", underline="single")

_thin = Side(style="thin", color="D9D9D9")
GRID  = Border(left=_thin, right=_thin, top=_thin, bottom=_thin)

DATE_FMT = "yyyy-mm-dd hh:mm:ss"
NUM_FMT  = "#,##0"

def fit(colmax, col, value):
    if value is None:
        return
    longest = max((len(s) for s in str(value).split("\n")), default=0)
    if longest > colmax.get(col, 0):
        colmax[col] = longest

def apply_widths(ws, colmax, min_w=6, max_w=55, pad=2):
    for col, n in colmax.items():
        ws.column_dimensions[get_column_letter(col)].width = max(min_w, min(max_w, n + pad))

def excel_safe(v):
    if v is None or isinstance(v, (int, float, bool)):
        return v
    if isinstance(v, (bytes, bytearray)):
        return "0x" + v.hex()[:256]
    if isinstance(v, (dt.datetime, dt.date, dt.time)):
        return v
    s = str(v)
    return s if len(s) <= 32767 else s[:32760] + "..."

def safe_sheet_name(base, used):
    name = (re.sub(r'[\\/?*\[\]:]', "_", base).strip() or "Sheet")[:31]
    cand, i = name, 1
    while cand.lower() in used:
        sfx = f"_{i}"; cand = name[:31 - len(sfx)] + sfx; i += 1
    used.add(cand.lower())
    return cand

def banner(ws, span, title, subtitle):
    ws.merge_cells(start_row=1, start_column=1, end_row=1, end_column=span)
    c = ws.cell(row=1, column=1, value=title)
    c.font = TITLE_FONT; c.alignment = Alignment(vertical="center", indent=1)
    for col in range(1, span + 1):
        ws.cell(row=1, column=col).fill = BANNER
    ws.row_dimensions[1].height = 30
    ws.merge_cells(start_row=2, start_column=1, end_row=2, end_column=span)
    s = ws.cell(row=2, column=1, value=subtitle)
    s.font = NOTE_FONT; s.alignment = Alignment(indent=1)

def write_header(ws, row, headers, colmax):
    for i, h in enumerate(headers, start=1):
        cell = ws.cell(row=row, column=i, value=h)
        cell.font = HEAD_FONT; cell.fill = HEAD_FILL; cell.border = GRID
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        fit(colmax, i, h)
    ws.row_dimensions[row].height = 24

wb = Workbook()

# ============================== Summary ==============================
ws = wb.active; ws.title = "Summary"
ws.sheet_properties.tabColor = NAVY
ws.sheet_view.showGridLines = False

HDR = ["#", "Schema", "Table", "Columns", "Rows", "Incremental Candidates",
       "Config Tuple (for generator)", "Sheet"]
ncol = len(HDR)
banner(ws, ncol, f"Pre-Ingestion Analysis   |   [{DATABASE}]",
       f"Server: {SERVER}      Generated: {datetime.now():%Y-%m-%d %H:%M}      "
       f"Top N sampled: {TOP_N}      Tables analysed: {len(analysis)}")

HROW = 4
colmax = {}
write_header(ws, HROW, HDR, colmax)
ws.freeze_panes = ws.cell(row=HROW + 1, column=1).coordinate

used_names = {"summary"}
table_sheet_map = []
r = HROW + 1
for idx, rec in enumerate(analysis, start=1):
    inc = [c["name"] for c in rec["columns"] if c.get("inc_conf")]
    cfg_tuple = f'("{DATABASE}", "{rec["schema"]}", "{rec["table"]}")'
    sname = safe_sheet_name(rec["table"], used_names)
    table_sheet_map.append((rec, sname))
    inc_text = ", ".join(inc) if inc else ("(error)" if rec["error"] else "(none found)")
    vals = [idx, rec["schema"], rec["table"], len(rec["columns"]),
            rec["row_count"] if rec["row_count"] is not None else "",
            inc_text, cfg_tuple, sname]
    for ci, v in enumerate(vals, start=1):
        cell = ws.cell(row=r, column=ci, value=v)
        cell.font = BASE_FONT; cell.border = GRID
        centered = ci in (1, 4, 5)
        cell.alignment = Alignment(vertical="center",
                                   horizontal="center" if centered else "left",
                                   wrap_text=(ci == 6), indent=0 if centered else 1)
        fit(colmax, ci, v)
    ws.cell(row=r, column=5).number_format = NUM_FMT
    fill = ERR_FILL if rec["error"] else (ZEBRA if idx % 2 == 0 else None)
    if fill:
        for ci in range(1, ncol + 1):
            ws.cell(row=r, column=ci).fill = fill
    link = ws.cell(row=r, column=8)
    link.hyperlink = f"#'{sname}'!A1"; link.font = LINK_FONT
    r += 1

apply_widths(ws, colmax)
ws.column_dimensions["A"].width = 5
ws.column_dimensions["F"].width = min(50, max(ws.column_dimensions["F"].width or 0, 30))
ws.column_dimensions["G"].width = min(48, max(ws.column_dimensions["G"].width or 0, 40))
ws.auto_filter.ref = f"A{HROW}:{get_column_letter(ncol)}{r - 1}"

# ============================ per-table sheets ============================
for rec, sname in table_sheet_map:
    ws = wb.create_sheet(sname)
    ws.sheet_view.showGridLines = False
    ws.sheet_properties.tabColor = "C00000" if rec["error"] else "BDD7EE"
    span = max(8, len(rec["headers"]) or 1)

    n_inc = sum(1 for c in rec["columns"] if c.get("inc_conf"))
    rows_txt = f'{rec["row_count"]:,}' if isinstance(rec["row_count"], int) else "?"
    banner(ws, span, f'{rec["schema"]}.{rec["table"]}',
           f'Columns: {len(rec["columns"])}      Rows: {rows_txt}      Incremental candidates: {n_inc}')

    if rec["error"]:
        ws.cell(row=4, column=1, value="Could not read this table:").font = LABEL_FONT
        ws.cell(row=5, column=1, value=rec["error"]).font = BASE_FONT
        ws.column_dimensions["A"].width = 80
        continue

    colmax = {}
    row = 4
    ws.cell(row=row, column=1, value="COLUMN INVENTORY").font = LABEL_FONT
    row += 1
    CINV = ["#", "Column", "Data Type", "Length / Prec", "Nullable",
            "Incremental?", "Confidence", "Why flagged"]
    write_header(ws, row, CINV, colmax)
    inv_header_row = row; row += 1
    for j, col in enumerate(rec["columns"]):
        conf = col.get("inc_conf")
        vals = [col["ordinal"], col["name"], col["type"], length_or_precision(col),
                col["nullable"], "YES" if conf else "", conf or "", col.get("inc_why") or ""]
        for ci, v in enumerate(vals, start=1):
            cell = ws.cell(row=row, column=ci, value=v)
            cell.font = BASE_FONT; cell.border = GRID
            centered = ci in (1, 5, 6, 7)
            cell.alignment = Alignment(vertical="center",
                                       horizontal="center" if centered else "left",
                                       indent=0 if centered else 1)
            fit(colmax, ci, v)
        if not conf and (j % 2 == 1):
            for ci in range(1, len(CINV) + 1):
                ws.cell(row=row, column=ci).fill = ZEBRA
        if conf:
            ws.cell(row=row, column=6).fill = CONF_FILL[conf]
            ws.cell(row=row, column=7).fill = CONF_FILL[conf]
            ws.cell(row=row, column=6).font = BOLD_SMALL
        row += 1

    row += 1
    ws.cell(row=row, column=1, value=f"TOP {TOP_N} ROWS  (sample)").font = LABEL_FONT
    row += 1
    if rec["sample_error"]:
        ws.cell(row=row, column=1, value="Could not sample rows: " + rec["sample_error"]).font = BASE_FONT
    elif rec["headers"]:
        write_header(ws, row, rec["headers"], colmax)
        row += 1
        for k, srow in enumerate(rec["sample"]):
            for ci, v in enumerate(srow, start=1):
                sv = excel_safe(v)
                cell = ws.cell(row=row, column=ci, value=sv)
                cell.font = BASE_FONT; cell.border = GRID
                is_num = isinstance(sv, (int, float)) and not isinstance(sv, bool)
                cell.alignment = Alignment(vertical="center",
                                           horizontal="right" if is_num else "left",
                                           indent=0 if is_num else 1)
                if isinstance(sv, (dt.datetime, dt.date)):
                    cell.number_format = DATE_FMT
                fit(colmax, ci, sv)
            if k % 2 == 1:
                for ci in range(1, len(rec["headers"]) + 1):
                    ws.cell(row=row, column=ci).fill = ZEBRA
            row += 1
    else:
        ws.cell(row=row, column=1, value="(no rows returned)").font = BASE_FONT

    apply_widths(ws, colmax)
    ws.column_dimensions["A"].width = max(5, ws.column_dimensions["A"].width or 5)
    ws.freeze_panes = ws.cell(row=inv_header_row + 1, column=1).coordinate

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
wb.save(OUTPUT_FILE)
print("wrote:", OUTPUT_FILE.resolve())

## 5 - Config tuples for the generator
Paste these straight into the `tables=[...]` list of your `PipelineConfig` in `sqlserver_to_snowflake.py`.

In [ ]:
ok      = [r for r in analysis if not r["error"]]
skipped = [r for r in analysis if r["error"]]

print("tables = [")
for rec in ok:
    print(f'    ("{DATABASE}", "{rec["schema"]}", "{rec["table"]}"),')
print("]")

if skipped:
    print("\n# skipped (errors):")
    for rec in skipped:
        print(f'#   {rec["schema"]}.{rec["table"]}: {rec["error"]}')

conn.close()
print("\nconnection closed.")